# Two-run comparison, year 2017

**Goal.** Quantify the difference between any two pyAPES result files over
2017, at 30-min resolution, for the wind/scalar profiles, ecosystem fluxes
and forest-floor variables the momentum solver and snow model feed into.

**Which two runs** is set in the single configuration cell below
(`FILE_A` / `FILE_B`, with `LABEL_A` / `LABEL_B` and optional per-file date
slices). All figures and tables report `B - A`, labelled from `LABEL_*`.

**Default pair**

- **A:** `results/FIHy_2012_2017.nc`, sliced to 2017 — old pyAPES, `fdm`
  momentum solver, old `DegreeDaySnow`.
- **B:** `results/FiHy_2017_new_U_fdm_degreeday.nc` — this repo, `fdm`
  solver, rewritten degree-day snow.

**Caveat — the default pair is not apples-to-apples.** `FIHy_2012_2017` is a
multi-year run started in 2012, so on 2017-01-01 it already carries a
spun-up state (SWE ~53 kg m$^{-2}$, frozen soil, equilibrated soil
moisture). The `FiHy_2017_*` runs cold-start on 2017-01-01 from the
parameter-file initial conditions (SWE 0, soil T 4 °C, GWL -2 m). Much of
the winter/spring difference below is this spinup mismatch, roughly common
to every `FiHy_2017_*` run, **not** an effect of the solver or snow switch.
To isolate the solver/snow effect, compare two `FiHy_2017_*` runs against
each other, or use an old-model run that also cold-starts 2017-01-01.

In [ ]:
%matplotlib widget
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xarray as xr

# make the repository importable regardless of which kernel/venv this notebook
# runs under (an editor's "fix import" quick-fix can otherwise rewrite the
# absolute import below into a broken relative one, e.g. `from ..pyAPES...`)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyAPES').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pyAPES.utils.iotools import read_results

sns.set_context('notebook')

RESULTS_DIR = REPO_ROOT / 'results'

# --- configuration: the two runs to compare -------------------------------
# every figure and table below reports B - A.
FILE_A = RESULTS_DIR / 'FIHy_2017_fluxnet_branch.nc'
FILE_B = RESULTS_DIR / 'FiHy_2017_new_U_fvm_fsm2.nc'
LABEL_A = 'old (fdm)'
LABEL_B = 'new fvm+fsm2'
# per-file date slice applied on load (None = whole file). Use this to pull
# 2017 out of a multi-year file; leave None for a file that is already 2017.
SLICE_A = ('2017-01-01', '2017-12-31')
SLICE_B = None
# -------------------------------------------------------------------------

DIFF_LABEL = f'{LABEL_B} - {LABEL_A}'


def _load(path, sl):
    ds = read_results(str(path))
    if sl is not None:
        ds = ds.sel(date=slice(*sl))
    return ds


old = _load(FILE_A, SLICE_A)   # 'A'
new = _load(FILE_B, SLICE_B)   # 'B'

sim = 0  # single-simulation files
print(f'A: {FILE_A.name}  [{LABEL_A}]  {old.sizes["date"]} timesteps, '
      f'{str(old.date.values[0])[:10]} ... {str(old.date.values[-1])[:10]}')
print(f'B: {FILE_B.name}  [{LABEL_B}]  {new.sizes["date"]} timesteps, '
      f'{str(new.date.values[0])[:10]} ... {str(new.date.values[-1])[:10]}')
print(f'figures/tables report:  {DIFF_LABEL}')

## Grid and time-axis consistency check

Before differencing anything, confirm the two runs share the same canopy
grid and the same set of timestamps for 2017. A mismatch here would make any
later difference meaningless.

In [ ]:
assert np.allclose(old.canopy_z.values, new.canopy_z.values), 'canopy grids differ'
assert np.allclose(old.soil_z.values, new.soil_z.values), 'soil grids differ'

# align on the intersection of timestamps (handles any off-by-one-step edge mismatch)
common_dates = np.intersect1d(old.date.values, new.date.values)
print(f'canopy grid: {len(old.canopy_z)} nodes, {float(old.canopy_z.min())} ... '
      f'{float(old.canopy_z.max())} m')
print(f'old dates: {old.sizes["date"]}, new dates: {new.sizes["date"]}, '
      f'common: {len(common_dates)}')

old = old.sel(date=common_dates)
new = new.sel(date=common_dates)
zc = old.canopy_z.values
assert old.sizes['date'] == new.sizes['date']
print('grids and time axis aligned:', old.sizes['date'], 'timesteps, dz =', zc[1] - zc[0], 'm')

# floor below which |new| is treated as zero for relative-difference plots
# (see the caveat under 'Helper functions' below), not a physical threshold
EPS_DIV = 1e-6


The canopy-profile variables are typically `NaN` at the final stored
timestep (`...-12-31T00:00:00`), an edge artifact of the run. All summary
statistics below use `nan`-aware reductions (`np.nanmean`,
`np.nanpercentile`, `np.nanmax`), so such timesteps are excluded rather than
propagating into every result as `NaN`.

## Helper functions

Four summary functions, two variable shapes times absolute/relative. In the
code `old` is run A and `new` is run B.

- **`profile_diff_summary`** / **`scalar_diff_summary`** compute the absolute
  difference `B - A` (`new - old`).
- **`profile_reldiff_summary`** / **`scalar_reldiff_summary`** compute the
  relative difference `(new - old) / new * 100 %`, pointwise, at every
  timestep (and every height for profiles) before reducing over time.

Both pairs reduce the same way: a mean, a max-|value|, and (for profiles) a
25th/75th percentile band per height, ready for `fill_betweenx`.

**Caveat on the relative-diff figures below**: dividing by `new` blows up
wherever `new` is at or near zero. Two places this happens on purpose:

- `canopy_wind_speed`/`canopy_friction_velocity` at the ground node
  (`z = 0`), where `U` is 0 (exactly so in the `fvm` scheme, near-0 in
  `fdm`) — division is undefined/ill-conditioned there, masked to `NaN`.
- Any flux that changes sign over the day (`canopy_NEE`, `canopy_GPP`, net
  CO2 exchange, `ffloor_ground_heat`, ...): near its zero-crossing a tiny
  denominator turns an unremarkable absolute difference into a huge
  percentage. The relative-diff panels for these variables are dominated by
  a few such spikes and are much less informative than their absolute
  counterparts above — read them together, not instead of.

In [ ]:
def profile_diff_summary(varname: str) -> dict:
    '''
    Time-reduced statistics of (new - old) for a canopy-profile variable.

    Args:
        varname (str): variable name, present in both datasets, dims
            (date, simulation, canopy)
    Returns:
        (dict): 'z' (array, [m]), 'diff' (array, [time, z], raw difference),
            'mean' (array, [z]), 'p25'/'p75' (array, [z]),
            'max_abs' (array, [z]), 'z_of_max' (float, height of the largest
            |diff| anywhere in the column), 'max_abs_overall' (float)
    '''
    a = old[varname].isel(simulation=sim).values   # [time, z]
    b = new[varname].isel(simulation=sim).values
    diff = b - a
    max_abs = np.nanmax(np.abs(diff), axis=0)
    iz = int(np.nanargmax(max_abs))
    return {'z': zc, 'diff': diff,
            'mean': np.nanmean(diff, axis=0),
            'p25': np.nanpercentile(diff, 25, axis=0),
            'p75': np.nanpercentile(diff, 75, axis=0),
            'max_abs': max_abs,
            'z_of_max': float(zc[iz]),
            'max_abs_overall': float(max_abs[iz])}


def scalar_diff_summary(varname: str) -> dict:
    '''
    Time-reduced statistics of (new - old) for a scalar time series.

    Args:
        varname (str): variable name, present in both datasets, dims
            (date, simulation)
    Returns:
        (dict): 'date' (array), 'diff' (array), 'mean', 'mean_abs', 'max_abs',
            'date_of_max' (timestamp of the largest |diff|)
    '''
    a = old[varname].isel(simulation=sim).values
    b = new[varname].isel(simulation=sim).values
    diff = b - a
    i = int(np.nanargmax(np.abs(diff)))
    return {'date': old.date.values, 'diff': diff,
            'mean': float(np.nanmean(diff)),
            'mean_abs': float(np.nanmean(np.abs(diff))),
            'max_abs': float(np.abs(diff[i])),
            'date_of_max': old.date.values[i]}


def plot_profile_diff(ax, summary: dict, label: str, color: str):
    '''Plots a fill_betweenx 25/75 band, the mean, and the max-|diff| envelope.'''
    ax.fill_betweenx(summary['z'], summary['p25'], summary['p75'],
                     color=color, alpha=0.25, label=f'{label}: 25-75th pct')
    ax.plot(summary['mean'], summary['z'], '-', color=color, lw=2, label=f'{label}: mean')
    ax.plot(summary['max_abs'], summary['z'], ':', color=color, lw=1.5,
           label=f'{label}: max |diff|')
    ax.axvline(0.0, color='0.4', ls=':', lw=1)
    ax.set_ylabel('z [m]')


def profile_reldiff_summary(varname: str) -> dict:
    '''
    Time-reduced statistics of (new - old) / new * 100 [%] for a canopy-profile
    variable. Division by (near-)zero new-values is masked to NaN rather than
    producing inf -- see the caveat above.

    Args: as profile_diff_summary.
    Returns: same shape as profile_diff_summary, values in percent.
    '''
    a = old[varname].isel(simulation=sim).values   # [time, z]
    b = new[varname].isel(simulation=sim).values
    reldiff = np.where(np.abs(b) > EPS_DIV, 100.0 * (b - a) / b, np.nan)
    max_abs = np.nanmax(np.abs(reldiff), axis=0)
    finite = np.isfinite(max_abs)
    iz = int(np.nanargmax(np.where(finite, max_abs, -np.inf)))
    return {'z': zc, 'diff': reldiff,
            'mean': np.nanmean(reldiff, axis=0),
            'p25': np.nanpercentile(reldiff, 25, axis=0),
            'p75': np.nanpercentile(reldiff, 75, axis=0),
            'max_abs': max_abs,
            'z_of_max': float(zc[iz]),
            'max_abs_overall': float(max_abs[iz])}


def scalar_reldiff_summary(varname: str) -> dict:
    '''
    Time-reduced statistics of (new - old) / new * 100 [%] for a scalar time
    series. Division by (near-)zero new-values is masked to NaN.

    Args: as scalar_diff_summary.
    Returns: same shape as scalar_diff_summary, values in percent.
    '''
    a = old[varname].isel(simulation=sim).values
    b = new[varname].isel(simulation=sim).values
    reldiff = np.where(np.abs(b) > EPS_DIV, 100.0 * (b - a) / b, np.nan)
    i = int(np.nanargmax(np.abs(reldiff)))
    return {'date': old.date.values, 'diff': reldiff,
            'mean': float(np.nanmean(reldiff)),
            'mean_abs': float(np.nanmean(np.abs(reldiff))),
            'max_abs': float(np.abs(reldiff[i])),
            'date_of_max': old.date.values[i]}


## 1. Wind and scalar profiles

`canopy_wind_speed`, `canopy_friction_velocity`, `canopy_h2o`, `canopy_co2`,
`canopy_temperature` — the five profile variables the momentum solver feeds
into, directly or through `Km`. The 25th/75th percentile band is the spread
of `new - old` across all of 2017 at each height; the dotted envelope is the
single worst timestep at each height, which is not necessarily the same
timestep at every height.

In [ ]:
PROFILE_VARS = {
    'canopy_wind_speed': 'U [m s$^{-1}$]',
    'canopy_friction_velocity': '$u_*$ [m s$^{-1}$]',
    'canopy_h2o': 'H2O [mol mol$^{-1}$]',
    'canopy_co2': 'CO2 [ppm]',
    'canopy_temperature': 'T [$^\\circ$C]',
}

profile_summaries = {v: profile_diff_summary(v) for v in PROFILE_VARS}

fig, axes = plt.subplots(1, len(PROFILE_VARS), figsize=(3 * len(PROFILE_VARS), 6), sharey=True)
for ax, (v, xlabel) in zip(axes, PROFILE_VARS.items()):
    plot_profile_diff(ax, profile_summaries[v], DIFF_LABEL, 'tab:blue')
    ax.set_xlabel(f'$\\Delta$ {xlabel}')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
for a, letter in zip(axes, 'abcde'):
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
fig.suptitle(f'{DIFF_LABEL}, year 2017: profile differences (25-75th pct band, mean, max |diff|)')
fig.tight_layout()

In [ ]:
rows = []
for v in PROFILE_VARS:
    s = profile_summaries[v]
    rows.append({'variable': v,
                 'max |diff| (overall)': s['max_abs_overall'],
                 'z of max [m]': s['z_of_max'],
                 'mean(diff) at z of max': float(np.interp(s['z_of_max'], s['z'], s['mean'])),
                 'mean(diff) column-avg': float(np.nanmean(s['mean'])),
                 'mean(|diff|) column-avg': float(np.nanmean(np.abs(s['diff']))),
                 'p25-p75 width, column-avg': float(np.mean(s['p75'] - s['p25']))})
profile_table = pd.DataFrame(rows).set_index('variable')
profile_table.round(5)

### 1b. Same, as relative difference

`(new - old) / new * 100 %`, same reduction and same panel layout as above.

In [ ]:
profile_reldiff_summaries = {v: profile_reldiff_summary(v) for v in PROFILE_VARS}

fig, axes = plt.subplots(1, len(PROFILE_VARS), figsize=(4 * len(PROFILE_VARS), 6), sharey=True)
for ax, (v, xlabel) in zip(axes, PROFILE_VARS.items()):
    plot_profile_diff(ax, profile_reldiff_summaries[v], DIFF_LABEL, 'tab:purple')
    ax.set_xlabel(f"relative $\\Delta$ {xlabel.split(' [')[0]} [%]")
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
for a, letter in zip(axes, 'abcde'):
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
fig.suptitle(f'{DIFF_LABEL}, year 2017: RELATIVE profile differences (25-75th pct band, mean, max |diff|)')
fig.tight_layout()

In [ ]:
rows = []
for v in PROFILE_VARS:
    s = profile_reldiff_summaries[v]
    rows.append({'variable': v,
                 'max |rel diff| [%] (overall)': s['max_abs_overall'],
                 'z of max [m]': s['z_of_max'],
                 'mean(rel diff) [%] column-avg': float(np.nanmean(s['mean'])),
                 'mean(|rel diff|) [%] column-avg': float(np.nanmean(np.abs(s['diff'])))})
profile_reldiff_table = pd.DataFrame(rows).set_index('variable')
profile_reldiff_table.round(3)

## 2. Ecosystem fluxes

`canopy_NEE`, `canopy_GPP`, `canopy_Reco`, `canopy_LE`, `canopy_SH`,
`canopy_Rnet`. There is no canopy-top "G" (ground heat flux) in this model —
G lives at the forest floor (`ffloor_ground_heat`), included here for
completeness since it was requested alongside the ecosystem fluxes.

In [ ]:
FLUX_VARS = {
    'canopy_NEE': 'NEE [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_GPP': 'GPP [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_Reco': 'R$_{eco}$ [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_LE': 'LE [W m$^{-2}$]',
    'canopy_SH': 'H [W m$^{-2}$]',
    'canopy_Rnet': 'R$_{net}$ [W m$^{-2}$]',
    'ffloor_ground_heat': 'G (forest floor) [W m$^{-2}$]',
}

flux_summaries = {v: scalar_diff_summary(v) for v in FLUX_VARS}

fig, axes = plt.subplots(4, 2, figsize=(13, 12), sharex=True)
ax = axes.ravel()
for a, (v, ylabel) in zip(ax, FLUX_VARS.items()):
    s = flux_summaries[v]
    a.plot(s['date'], s['diff'], '-', color='tab:red', lw=0.6)
    a.axhline(0.0, color='0.4', ls=':', lw=1)
    a.set_ylabel(f'$\\Delta$ {ylabel}')
ax[-1].axis('off')
for a, letter in zip(ax, 'abcdefg'):
    a.text(0.02, 0.95, f'{letter})', transform=a.transAxes, va='top')
fig.suptitle(f'{DIFF_LABEL}, year 2017: ecosystem flux differences')
fig.tight_layout()

In [ ]:
rows = []
for v in FLUX_VARS:
    s = flux_summaries[v]
    rows.append({'variable': v,
                 'max |diff|': s['max_abs'],
                 'date of max': str(s['date_of_max'])[:16],
                 'mean(diff)': s['mean'],
                 'mean(|diff|)': s['mean_abs']})
flux_table = pd.DataFrame(rows).set_index('variable')
flux_table.round(5)

### 2b. Same, as relative difference

`(new - old) / new * 100 %`. NEE, GPP and net CO2 exchange cross zero every
day, so expect this panel to be dominated by zero-crossing spikes — read it
next to the absolute-diff figure above, not instead of it.

In [ ]:
flux_reldiff_summaries = {v: scalar_reldiff_summary(v) for v in FLUX_VARS}

fig, axes = plt.subplots(4, 2, figsize=(13, 12), sharex=True)
ax = axes.ravel()
for a, (v, ylabel) in zip(ax, FLUX_VARS.items()):
    s = flux_reldiff_summaries[v]
    a.plot(s['date'], s['diff'], '-', color='tab:purple', lw=0.6)
    a.axhline(0.0, color='0.4', ls=':', lw=1)
    a.set_ylabel(f"relative $\\Delta$ {ylabel.split(' [')[0]} [%]")
    a.set_yscale('log')
ax[-1].axis('off')
for a, letter in zip(ax, 'abcdefg'):
    a.text(0.02, 0.95, f'{letter})', transform=a.transAxes, va='top')
fig.suptitle(f'{DIFF_LABEL}, year 2017: RELATIVE ecosystem flux differences')
fig.tight_layout()

In [ ]:
rows = []
for v in FLUX_VARS:
    s = flux_reldiff_summaries[v]
    rows.append({'variable': v,
                 'max |rel diff| [%]': s['max_abs'],
                 'date of max': str(s['date_of_max'])[:16],
                 'mean(rel diff) [%]': s['mean'],
                 'mean(|rel diff|) [%]': s['mean_abs']})
flux_reldiff_table = pd.DataFrame(rows).set_index('variable')
flux_reldiff_table.round(3)

## 3. Forest floor

Every `ffloor_*` scalar in the output schema: energy balance, water balance,
carbon exchange, and state.

In [ ]:
FFLOOR_VARS = {
    'ffloor_surface_temperature': 'T$_{surf}$ [$^\\circ$C]',
    'ffloor_net_radiation': 'R$_{net}$ [W m$^{-2}$]',
    'ffloor_sensible_heat': 'H [W m$^{-2}$]',
    'ffloor_latent_heat': 'LE [W m$^{-2}$]',
    'ffloor_ground_heat': 'G [W m$^{-2}$]',
    'ffloor_energy_closure': 'energy closure err. [W m$^{-2}$]',
    'ffloor_evaporation': 'evaporation [kg m$^{-2}$ s$^{-1}$]',
    'ffloor_soil_evaporation': 'soil evaporation [kg m$^{-2}$ s$^{-1}$]',
    'ffloor_net_co2': 'net CO2 [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'ffloor_respiration': 'respiration [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'ffloor_soil_respiration': 'soil respiration [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'ffloor_water_storage': 'water storage [kg m$^{-2}$]',
    'ffloor_snow_water_equivalent': 'SWE [kg m$^{-2}$]',
}

ffloor_summaries = {v: scalar_diff_summary(v) for v in FFLOOR_VARS if v in old and v in new}
missing = [v for v in FFLOOR_VARS if v not in ffloor_summaries]
if missing:
    print('not present in one of the two files, skipped:', missing)

n = len(ffloor_summaries)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3 * nrows), sharex=True)
ax = axes.ravel()
for a, (v, ylabel) in zip(ax, ((v, FFLOOR_VARS[v]) for v in ffloor_summaries)):
    s = ffloor_summaries[v]
    a.plot(s['date'], s['diff'], '-', color='tab:green', lw=0.6)
    a.axhline(0.0, color='0.4', ls=':', lw=1)
    a.set_ylabel(f'$\\Delta$ {ylabel}', fontsize=8)
for a in ax[n:]:
    a.axis('off')
for a, letter in zip(ax, 'abcdefghijklm'):
    a.text(0.02, 0.95, f'{letter})', transform=a.transAxes, va='top', fontsize=8)
fig.suptitle(f'{DIFF_LABEL}, year 2017: forest floor differences')
fig.tight_layout()

In [ ]:
rows = []
for v, s in ffloor_summaries.items():
    rows.append({'variable': v,
                 'max |diff|': s['max_abs'],
                 'date of max': str(s['date_of_max'])[:16],
                 'mean(diff)': s['mean'],
                 'mean(|diff|)': s['mean_abs']})
ffloor_table = pd.DataFrame(rows).set_index('variable')
ffloor_table.round(6)

### 3b. Same, as relative difference

`(new - old) / new * 100 %`.

In [ ]:
ffloor_reldiff_summaries = {v: scalar_reldiff_summary(v) for v in ffloor_summaries}

n = len(ffloor_reldiff_summaries)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3 * nrows), sharex=True)
ax = axes.ravel()
for a, (v, ylabel) in zip(ax, ((v, FFLOOR_VARS[v]) for v in ffloor_reldiff_summaries)):
    s = ffloor_reldiff_summaries[v]
    a.plot(s['date'], s['diff'], '-', color='tab:purple', lw=0.6)
    a.axhline(0.0, color='0.4', ls=':', lw=1)
    a.set_ylabel(f"rel. $\\Delta$ {ylabel.split(' [')[0]} [%]", fontsize=8)
    a.set_yscale('log')
for a in ax[n:]:
    a.axis('off')
for a, letter in zip(ax, 'abcdefghijklm'):
    a.text(0.02, 0.95, f'{letter})', transform=a.transAxes, va='top', fontsize=8)
fig.suptitle(f'{DIFF_LABEL}, year 2017: RELATIVE forest floor differences')
fig.tight_layout()

In [ ]:
rows = []
for v, s in ffloor_reldiff_summaries.items():
    rows.append({'variable': v,
                 'max |rel diff| [%]': s['max_abs'],
                 'date of max': str(s['date_of_max'])[:16],
                 'mean(rel diff) [%]': s['mean'],
                 'mean(|rel diff|) [%]': s['mean_abs']})
ffloor_reldiff_table = pd.DataFrame(rows).set_index('variable')
ffloor_reldiff_table.round(3)

## 4. Summary: everything in one table

Relative change is `mean(|diff|) / mean(|A|)`, computed over 2017, so
variables with very different natural magnitudes (ppm CO2 vs. mmol H2O vs.
W m$^{-2}$) are comparable on one axis. Profile variables are column-averaged
first. `diff` is `B - A`.

In [ ]:
summary_rows = []

for v in PROFILE_VARS:
    s = profile_summaries[v]
    old_mag = float(np.nanmean(np.abs(old[v].isel(simulation=sim).values)))
    summary_rows.append({'variable': v, 'kind': 'profile',
                         'max |diff|': s['max_abs_overall'],
                         'mean(diff)': float(np.nanmean(s['mean'])),
                         'mean(|diff|)': float(np.nanmean(np.abs(s['diff']))),
                         'rel. change [%]': 100 * float(np.nanmean(np.abs(s['diff']))) / (old_mag + 1e-12)})

for v in list(FLUX_VARS) + [k for k in FFLOOR_VARS if k not in FLUX_VARS and k in ffloor_summaries]:
    s = flux_summaries.get(v) or ffloor_summaries.get(v)
    old_mag = float(np.nanmean(np.abs(old[v].isel(simulation=sim).values)))
    summary_rows.append({'variable': v, 'kind': 'scalar',
                         'max |diff|': s['max_abs'],
                         'mean(diff)': s['mean'],
                         'mean(|diff|)': s['mean_abs'],
                         'rel. change [%]': 100 * s['mean_abs'] / (old_mag + 1e-12)})

summary_table = pd.DataFrame(summary_rows).set_index('variable')
summary_table.sort_values('rel. change [%]', ascending=False).round(4)

## 5. Time series of both runs (absolute values, not differences)

The sections above only show `B - A`. Here are the raw 2017 time series of
each run overlaid, run A (`LABEL_A`) and run B (`LABEL_B`) on the same axes,
so the actual magnitude and seasonal/diurnal variation is visible, not just
the residual.

- **Figure 5a — ecosystem fluxes:** NEE, GPP, R$_{eco}$, LE, H, and G
  (forest-floor ground heat), one stacked panel each.
- **Figure 5b — forest floor:** every `ffloor_*` scalar present in both
  files, one stacked panel each.

In [ ]:
def plot_series_overlay(varnames: dict, letters: str, title: str,
                        panel_height: float = 2.6):
    '''
    Stacked time-series panels, one per variable, with run A and run B
    overlaid on the same axes.

    Args:
        varnames (dict): {variable name: y-axis label}; variables must be
            scalar (date, simulation) and present in both `old` and `new`.
        letters (str): subplot letters, one char per variable.
        title (str): figure suptitle.
        panel_height (float): height in inches per stacked panel.
    '''
    items = [(v, lab) for v, lab in varnames.items() if v in old and v in new]
    n = len(items)
    fig, axes = plt.subplots(n, 1, figsize=(15, panel_height * n), sharex=True)
    axes = np.atleast_1d(axes)
    for a, (v, ylabel), letter in zip(axes, items, letters):
        a.plot(old.date.values, old[v].isel(simulation=sim).values,
               '-', color='tab:blue', lw=0.5, label=LABEL_A)
        a.plot(new.date.values, new[v].isel(simulation=sim).values,
               '-', color='tab:orange', lw=0.5, alpha=0.8, label=LABEL_B)
        a.axhline(0.0, color='0.4', ls=':', lw=1)
        a.set_ylabel(ylabel, fontsize=9)
        a.text(0.005, 0.95, f'{letter})', transform=a.transAxes, va='top', fontsize=9)
    axes[0].legend(frameon=False, fontsize=9, loc='upper right', ncol=2)
    axes[-1].set_xlabel('2017')
    fig.suptitle(title)
    fig.tight_layout()
    return fig


In [ ]:
# Figure 5a: ecosystem fluxes, both runs overlaid
SERIES_FLUX_VARS = {
    'canopy_NEE': 'NEE [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_GPP': 'GPP [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_Reco': 'R$_{eco}$ [$\\mu$mol m$^{-2}$ s$^{-1}$]',
    'canopy_LE': 'LE [W m$^{-2}$]',
    'canopy_SH': 'H [W m$^{-2}$]',
    'ffloor_ground_heat': 'G (forest floor) [W m$^{-2}$]',
}

fig_5a = plot_series_overlay(
    SERIES_FLUX_VARS, 'abcdef',
    f'Ecosystem fluxes, year 2017: {LABEL_A} vs {LABEL_B}')


In [ ]:
# Figure 5b: forest-floor scalars, both runs overlaid
fig_5b = plot_series_overlay(
    FFLOOR_VARS, 'abcdefghijklm',
    f'Forest floor, year 2017: {LABEL_A} vs {LABEL_B}',
    panel_height=2.2)


## 6. Comparison against eddy-covariance measurements

Both runs are scored against the measured Hyytiälä EC fluxes
(`FI_HYY_EC_PATH` in `.env`, file `FIHy_flx_*.dat`). Processing mirrors
`~/pyAPES/fluxnet_analysis/diurnal_comparison_utils.py`:

- **Quality control — only quality-corrected half-hours are used.** Where the
  file carries a QC flag (`Qc_NEE` for NEE/GPP/Reco, `Qc_H` for H, `Qc_ET`
  for LE) every half-hour with `QC != 0` is set to `NaN` before any metric or
  plot. `Gflux` has no QC flag at this site (and is mostly missing).
- **Variable map:** `NEE→canopy_NEE`, `GPP→canopy_GPP`, `Reco→canopy_Reco`,
  `H→canopy_SH`, `LE→canopy_LE`, `G→ffloor_ground_heat`.
- **Common paired sample.** For each flux a single 30-min dataframe holds EC,
  run A and run B; any half-hour missing in EC or in either run is dropped,
  so both runs are scored on the exact same timestamps and Fig. 6a's three
  daily-mean lines share one denominator.
- **Metrics** — `R²` (scikit-learn `r2_score`, EC as reference, can go
  negative), `RMSE`, `bias = mean(model − EC)`, and the OLS line
  `EC = a·model + b` — are computed on those matched 30-min pairs. The
  time-series panels (Fig. 6a) show daily means for legibility; the scatter
  (Fig. 6b) shows the 30-min pairs, coloured by run, with the 1:1 line and
  each run's fitted line.

In [ ]:
import os
from dotenv import load_dotenv
from scipy import stats
from sklearn.metrics import r2_score, root_mean_squared_error

load_dotenv(REPO_ROOT / '.env')

# EC comparison variable -> (EC column, QC column or None, model netCDF var, axis label)
EC_MAP = {
    'NEE':  ('NEE',   'Qc_NEE', 'canopy_NEE',         'NEE [$\\mu$mol m$^{-2}$ s$^{-1}$]'),
    'GPP':  ('GPP',   'Qc_NEE', 'canopy_GPP',         'GPP [$\\mu$mol m$^{-2}$ s$^{-1}$]'),
    'Reco': ('Reco',  'Qc_NEE', 'canopy_Reco',        'R$_{eco}$ [$\\mu$mol m$^{-2}$ s$^{-1}$]'),
    'LE':   ('LE',    'Qc_ET',  'canopy_LE',          'LE [W m$^{-2}$]'),
    'H':    ('H',     'Qc_H',   'canopy_SH',          'H [W m$^{-2}$]'),
    'G':    ('Gflux', None,     'ffloor_ground_heat', 'G [W m$^{-2}$]'),
}

ec_file = Path(os.getenv('FI_HYY_EC_PATH'))
if not ec_file.exists():                       # the .env path may omit the FI-Hyy/ subdir
    alt = ec_file.parent / 'FI-Hyy' / ec_file.name
    ec_file = alt if alt.exists() else ec_file
print('EC file:', ec_file, '| exists:', ec_file.exists())

raw_ec = pd.read_csv(ec_file, sep=';').replace(-9999, np.nan)
raw_ec['datetime'] = pd.to_datetime(raw_ec[['year', 'month', 'day', 'hour', 'minute']])
raw_ec = raw_ec.set_index('datetime')

# put EC on the model's 30-min time axis (both runs share it after the alignment cell)
model_dates = pd.DatetimeIndex(old.date.values)
ec_on_grid = raw_ec.reindex(model_dates)


def _model_series(ds, ncname):
    return pd.Series(ds[ncname].isel(simulation=sim).values, index=model_dates)


ec_matched, modA_matched, modB_matched = {}, {}, {}
for key, (eccol, qccol, ncname, _lab) in EC_MAP.items():
    meas = ec_on_grid[eccol].copy()
    if qccol is not None and qccol in ec_on_grid:
        meas[ec_on_grid[qccol] != 0] = np.nan          # keep only quality-corrected half-hours
    ec_matched[key] = meas
    modA_matched[key] = _model_series(old, ncname)
    modB_matched[key] = _model_series(new, ncname)

print('quality-corrected EC half-hours in 2017:',
      {k: int(ec_matched[k].notna().sum()) for k in EC_MAP})


def score(measured, modeled) -> dict:
    '''
    Skill metrics of a model series against EC, on the finite paired values.

    R2 and RMSE come from scikit-learn (r2_score, root_mean_squared_error);
    the OLS fit EC = slope*model + intercept from scipy.stats.linregress; bias
    is mean(model - EC).
    '''
    m = np.asarray(measured, float)
    p = np.asarray(modeled, float)
    ok = np.isfinite(m) & np.isfinite(p)
    m, p = m[ok], p[ok]
    if len(m) < 2:
        return dict(r2=np.nan, rmse=np.nan, bias=np.nan,
                    slope=np.nan, intercept=np.nan, n=len(m))
    fit = stats.linregress(p, m)                          # x = model, y = EC
    return dict(r2=float(r2_score(m, p)),                 # y_true = EC, y_pred = model
                rmse=float(root_mean_squared_error(m, p)),
                bias=float(np.mean(p - m)),
                slope=float(fit.slope), intercept=float(fit.intercept), n=int(len(m)))


In [ ]:
# Figure 6a: flux time series, EC vs both runs
# One common 30-min dataframe per flux (EC, run A, run B). Half-hours that are
# missing in EC or in either run are dropped so all three series rest on the
# exact same timestamps; only then are daily means and the metrics computed.
ec_scores = {}

fig, axes = plt.subplots(len(EC_MAP), 1, figsize=(15, 2.9 * len(EC_MAP)), sharex=True)
for a, (key, (eccol, qccol, ncname, lab)), letter in zip(axes, EC_MAP.items(), 'abcdef'):
    paired = pd.DataFrame({'ec': ec_matched[key],
                           'A': modA_matched[key],
                           'B': modB_matched[key]}).dropna()
    daily = paired.resample('D').mean()

    a.plot(daily.index, daily['ec'].values, '-', color='k', lw=1.3, label='EC (QC)')
    a.plot(daily.index, daily['A'].values, '-', color='tab:blue', lw=1.0, alpha=0.9, label=LABEL_A)
    a.plot(daily.index, daily['B'].values, '-', color='tab:orange', lw=1.0, alpha=0.9, label=LABEL_B)
    a.axhline(0.0, color='0.5', ls=':', lw=1)
    a.set_ylabel(lab, fontsize=9)

    sA = score(paired['ec'].values, paired['A'].values)
    sB = score(paired['ec'].values, paired['B'].values)
    ec_scores[key] = {'A': sA, 'B': sB}
    txt = (f"{LABEL_A}:  R$^2$={sA['r2']:.2f}  RMSE={sA['rmse']:.2f}  bias={sA['bias']:+.2f}\n"
           f"{LABEL_B}:  R$^2$={sB['r2']:.2f}  RMSE={sB['rmse']:.2f}  bias={sB['bias']:+.2f}"
           f"   (30-min, n={sA['n']})")
    a.text(0.004, 0.96, f'{letter})', transform=a.transAxes, va='top', fontsize=9, fontweight='bold')
    a.text(0.055, 0.97, txt, transform=a.transAxes, va='top', fontsize=7.5,
           bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

axes[0].legend(frameon=False, fontsize=8, loc='upper right', ncol=3)
axes[-1].set_xlabel('2017')
fig.suptitle('Ecosystem fluxes vs eddy covariance, Hyytiälä 2017 '
             '(daily means; R$^2$/RMSE/bias on 30-min paired, QC-filtered values)')
fig.tight_layout()


In [ ]:
# Figure 6b: model (x) vs EC (y) scatter, 30-min QC pairs, colour = run
# Same common per-flux dataframe as Fig. 6a: EC, run A and run B on identical
# 30-min timestamps, rows with any NaN dropped. R2/RMSE/bias and the fitted
# line are computed on these 30-min pairs (not on daily means).
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for a, (key, (eccol, qccol, ncname, lab)), letter in zip(axes.ravel(), EC_MAP.items(), 'abcdef'):
    paired = pd.DataFrame({'ec': ec_matched[key],
                           'A': modA_matched[key],
                           'B': modB_matched[key]}).dropna()
    sA = score(paired['ec'].values, paired['A'].values)
    sB = score(paired['ec'].values, paired['B'].values)
    ec_scores[key] = {'A': sA, 'B': sB}

    y = paired['ec'].values
    lims = []
    for run_label, col, s, color in ((LABEL_A, 'A', sA, 'tab:blue'),
                                     (LABEL_B, 'B', sB, 'tab:orange')):
        x = paired[col].values
        a.scatter(x, y, s=3, alpha=0.12, color=color, label=run_label, rasterized=True)
        if len(x):
            lims += [np.nanpercentile(x, [0.5, 99.5]), np.nanpercentile(y, [0.5, 99.5])]
        if np.isfinite(s['slope']) and len(x):
            xline = np.array([x.min(), x.max()])
            a.plot(xline, s['slope'] * xline + s['intercept'], '-', color=color, lw=1.6)

    # square axes clipped to the robust (0.5-99.5 pct) data range so the cloud,
    # the fitted lines and the 1:1 line are all readable
    lo = float(np.min([p[0] for p in lims])) if lims else 0.0
    hi = float(np.max([p[1] for p in lims])) if lims else 1.0
    a.plot([lo, hi], [lo, hi], 'k--', lw=1, label='1:1')
    a.set_xlim(lo, hi)
    a.set_ylim(lo, hi)
    a.set_aspect('equal', 'box')
    a.set_xlabel(f'model  {lab}', fontsize=8)
    a.set_ylabel(f'EC  {lab}', fontsize=8)

    txt = (f"{LABEL_A}: R$^2$={sA['r2']:.2f} RMSE={sA['rmse']:.2f} bias={sA['bias']:+.2f}\n"
           f"   EC = {sA['slope']:.2f}·model {sA['intercept']:+.2f}\n"
           f"{LABEL_B}: R$^2$={sB['r2']:.2f} RMSE={sB['rmse']:.2f} bias={sB['bias']:+.2f}\n"
           f"   EC = {sB['slope']:.2f}·model {sB['intercept']:+.2f}\n"
           f"   n={sA['n']}")
    a.text(0.03, 0.99, f'{letter})', transform=a.transAxes, va='top', fontweight='bold')
    a.text(0.03, 0.92, txt, transform=a.transAxes, va='top', fontsize=6.8,
           bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.9))
    a.legend(loc='lower right', fontsize=7, markerscale=3, framealpha=0.9)

fig.suptitle('Model vs eddy covariance (30-min, QC-filtered), colour = run, Hyytiälä 2017')
fig.tight_layout()
